# Poisoned Model Reference

This notebook generates detections from the poisoned model as a reference submission in a competition format 

**Workflow**
1. Install Detectron2 + deps
2. Load the **poisoned** RetinaNet model
3. Run inference on the test set
4. Save `submission.csv`

## 1. Install Detectron2

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 6.7 MB/s eta 0:00:00


## 2. Imports

In [2]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

## 3. Configuration

The **model-architecture** block must match how the poisoned model was trained; changing those values will cause silent weight mismatches (new random detection heads) and your submission will score poorly.

In [3]:
# ── Input paths (Kaggle Datasets) ──
POISONED_WEIGHTS = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
TEST_DIR         = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"

# ── Output paths ──
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# ── Model architecture (MUST match the poisoned model's training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Inference ──
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

## 4. 16-bit image handling

The input images are **uint16 single-channel PNGs**. To be compatible with the Detectron2 framework without losing the original numerical precision, we scale them to float32 in 0–255 range. We also override Detectron2's default `DatasetMapper` to:
- Read with `IMREAD_UNCHANGED`
- Duplicate the single channel to 3 channels (Detectron2 expects BGR-like 3-channel input)

In [4]:
class UInt16DatasetMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 in [0, 255] and attaches empty instances (the unlearning signal)."""
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)

        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

## 5. Config, Inference & Submission

Builds the Detectron2 `CfgNode` from the base RetinaNet recipe, overrides anchors + weights to match the poisoned model. Iterate over test images, keep detections with `confidence >= CONF_THRESH`, and assemble each image's `prediction_string` as `"conf x y w h conf x y w h ..."` (space-delimited, see the submission format in the Overview tab).

In [5]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))

cfg.MODEL.WEIGHTS = POISONED_WEIGHTS
cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES

predictor = DefaultPredictor(cfg)


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])

    # Use a single space for empty predictions so Kaggle's null-check passes.
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


Running inference on 2000 images...


Inference:   0%|          | 0/2000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Inference: 100%|██████████| 2000/2000 [04:00<00:00,  8.31it/s]

Wrote /kaggle/working/submission.csv  (2000 rows)


,id,image_id,prediction_string
0,0,0,0.708409 893.96 186.20 9.06 40.97 0.602567 208...
1,1,1,0.207372 541.47 428.22 25.57 10.42
2,2,10,0.914616 4.98 17.87 37.53 58.97 0.375197 28.86...
3,3,100,0.491582 997.92 769.79 21.76 52.10 0.390500 58...
4,4,1000,0.252598 1008.89 582.88 8.93 39.36
